# DetectiveDB — Caso 001: O Relogio Parado
## Laboratorio de Engenharia de Dados com Python, DuckDB e dbt Core

Este notebook demonstra de ponta a ponta o pipeline do **DetectiveDB**:
1. Configuracao de ambiente e instalacao de dependencias (`dbt-core`, `dbt-duckdb`, `duckdb`, `pandas`).
2. Ingestao de cinco arquivos CSV na camada `raw` do DuckDB.
3. Validacao de conexao (`dbt debug`), transformacao modular (`dbt run`) e auditoria de qualidade (`dbt test`).
4. Experimento controlado de qualidade: injecao proposital de duplicidade em `raw.evidence`, captura da falha real no teste `unique` e restauracao da fonte.
5. Consulta analitica final sobre `mart_case_summary` e `fct_testimonies`.

### 1. Instalacao de Dependencias
No Google Colab (ou em um ambiente virtual limpo), execute a celula abaixo para instalar as versoes validadas do projeto.

In [1]:
# Descomente a linha abaixo ao executar no Google Colab:
# !pip install -q dbt-core==1.12.7 dbt-duckdb==1.11.1 duckdb==2.0.0 pandas==3.1.1

import importlib.metadata as md

pkgs = ['dbt-core', 'dbt-duckdb', 'duckdb', 'pandas']
versions = ', '.join(f'{p}=={md.version(p)}' for p in pkgs)
print(f'[OK] Dependencias verificadas: {versions}')

[OK] Dependencias verificadas: dbt-core==1.12.7, dbt-duckdb==1.11.1, duckdb==2.0.0, pandas==3.1.1


### 2. Configuracao de Caminhos e Obtencao do Repositorio
Esta celula isola os caminhos do projeto para funcionar tanto localmente (a partir de `notebooks/` ou da raiz) quanto no Google Colab (via clone configuravel ou upload do diretorio do projeto).

In [2]:
import os
import subprocess
import sys
from pathlib import Path

# Se estiver no Google Colab e desejar clonar seu repositorio, defina REPO_GIT_URL:
REPO_GIT_URL = os.environ.get('DETECTIVEDB_GIT_URL', '')

cwd = Path.cwd().resolve()
if (cwd / 'detective_dbt').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'detective_dbt').exists():
    PROJECT_ROOT = cwd.parent
elif REPO_GIT_URL:
    target_dir = cwd / 'detectivedb'
    if not target_dir.exists():
        subprocess.run(['git', 'clone', REPO_GIT_URL, str(target_dir)], check=True)
    PROJECT_ROOT = target_dir
else:
    raise FileNotFoundError(
        'Estrutura do projeto nao encontrada. No Colab, faca upload da pasta do projeto '
        'ou defina a variavel DETECTIVEDB_GIT_URL antes de executar.'
    )

DBT_DIR = PROJECT_ROOT / 'detective_dbt'
DB_PATH = PROJECT_ROOT / 'data' / 'detectivedb.duckdb'
SCRIPTS_DIR = PROJECT_ROOT / 'scripts'

print(f'[INFO] Raiz do projeto: {PROJECT_ROOT}')
print(f'[INFO] Diretorio dbt:   {DBT_DIR}')
print(f'[INFO] Banco DuckDB:    {DB_PATH}')

[INFO] Raiz do projeto: C:\Projs\Proj_DBT
[INFO] Diretorio dbt:   C:\Projs\Proj_DBT\detective_dbt
[INFO] Banco DuckDB:    C:\Projs\Proj_DBT\data\detectivedb.duckdb


### 3. Ingestao da Camada Bruta (`raw`) no DuckDB
Executamos `scripts/ingest_raw.py` para criar o schema `raw` e carregar os cinco arquivos CSV (`suspects.csv`, `evidence.csv`, `testimonies.csv`, `cameras.csv`, `locations.csv`).

In [3]:
def run_cmd(cmd: list[str], cwd: Path = PROJECT_ROOT, check: bool = True) -> subprocess.CompletedProcess:
    result = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'Comando falhou com codigo {result.returncode}: {cmd}')
    return result

run_cmd([sys.executable, str(SCRIPTS_DIR / 'ingest_raw.py')], cwd=PROJECT_ROOT);

[INFO] Iniciando ingestao da camada raw no DuckDB...
[INFO] Banco alvo: C:\Projs\Proj_DBT\data\detectivedb.duckdb
[OK] Tabela raw.suspects carregada com 8 registros.
[OK] Tabela raw.evidence carregada com 20 registros.
[OK] Tabela raw.testimonies carregada com 14 registros.
[OK] Tabela raw.cameras carregada com 28 registros.
[OK] Tabela raw.locations carregada com 8 registros.
[OK] Ingestao concluida com sucesso.


### 4. Estado Valido: `dbt debug`, `dbt run` e `dbt test`
Com os dados brutos integros, validamos a conexao do profile DuckDB, construimos as views `staging` e tabelas `marts`, e executamos os 96 testes de qualidade (incluindo `unique`, `not_null`, `relationships`, `accepted_values` e os 2 testes SQL personalizados).

In [4]:
dbt_executable = str(Path(sys.executable).parent / ('dbt.exe' if os.name == 'nt' else 'dbt'))
if not Path(dbt_executable).exists():
    dbt_executable = 'dbt'

print('=== [1/3] dbt debug ===')
run_cmd([dbt_executable, 'debug', '--profiles-dir', '.'], cwd=DBT_DIR)

print('\n=== [2/3] dbt run ===')
run_cmd([dbt_executable, 'run', '--profiles-dir', '.'], cwd=DBT_DIR)

print('\n=== [3/3] dbt test ===')
run_cmd([dbt_executable, 'test', '--profiles-dir', '.'], cwd=DBT_DIR);

=== [1/3] dbt debug ===
Connection test: [OK connection ok]
All checks passed!

=== [2/3] dbt run ===
Finished running 6 table models, 5 view models in 0 hours 0 minutes and 0.76 seconds (0.76s).
Completed successfully
Done. PASS=11 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=11

=== [3/3] dbt test ===
Finished running 96 data tests in 0 hours 0 minutes and 3.74 seconds (3.74s).
Completed successfully
Done. PASS=96 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=96


### 5. Estado Invalido: Injecao de Duplicidade e Falha Esperada no `dbt test`
Agora executamos `scripts/inject_bad_data.py`, que insere uma linha duplicada com `evidence_id = 'E001'` diretamente em `raw.evidence`.

**Por que a falha ocorre?**
- Em `detective_dbt/models/sources.yml`, a coluna `evidence_id` da fonte `raw.evidence` e do modelo `stg_evidence` possuem o teste `unique`.
- A camada `staging` nao aplica deduplicacao silenciosa (`DISTINCT`), preservando a fidelidade da auditoria.
- Ao rodar `dbt test`, a consulta gerada pelo dbt encontra 1 chave com mais de uma ocorrencia (`E001`) e retorna codigo de erro (`ERROR=2`).

In [5]:
run_cmd([sys.executable, str(SCRIPTS_DIR / 'inject_bad_data.py')], cwd=PROJECT_ROOT)

print('\n=== Executando dbt test sobre a fonte contaminada (falha esperada) ===')
bad_test = run_cmd(
    [dbt_executable, 'test', '--profiles-dir', '.', '--select', 'source:raw.evidence', 'stg_evidence'],
    cwd=DBT_DIR,
    check=False,
)
assert bad_test.returncode != 0, 'O teste deveria ter falhado com a duplicidade injetada!'
print(f'[OK] Falha de unicidade capturada conforme esperado (returncode={bad_test.returncode}).')

[INFO] Injetando duplicidade proposital em raw.evidence (evidence_id = 'E001')...
[ALERTA] Anomalia injetada: raw.evidence possui agora 21 linhas (2 ocorrencias para evidence_id = 'E001').
[INFO] Execute `dbt test --profiles-dir .` em `detective_dbt/` para verificar a falha.

=== Executando dbt test sobre a fonte contaminada (falha esperada) ===
Failure in test source_unique_raw_evidence_evidence_id (models\sources.yml)
  Got 1 result, configured to fail if != 0
Failure in test unique_stg_evidence_evidence_id (models\sources.yml)
  Got 1 result, configured to fail if != 0
Done. PASS=13 WARN=0 ERROR=2 SKIP=0 NO-OP=0 REUSED=0 TOTAL=15
[OK] Falha de unicidade capturada conforme esperado (returncode=1).


### 6. Recuperacao: Restauracao da Fonte e Revalidacao Completa
Executamos `scripts/reset_demo.py` para recarregar os CSVs originais sem duplicidades e rodamos novamente `dbt run` e `dbt test` para comprovar o retorno ao estado integro (`PASS=96`).

In [6]:
run_cmd([sys.executable, str(SCRIPTS_DIR / 'reset_demo.py')], cwd=PROJECT_ROOT)

print('\n=== Reexecutando dbt run e dbt test apos restauracao ===')
run_cmd([dbt_executable, 'run', '--profiles-dir', '.'], cwd=DBT_DIR)
run_cmd([dbt_executable, 'test', '--profiles-dir', '.'], cwd=DBT_DIR);

[INFO] Restaurando camada raw a partir dos arquivos CSV originais...
[OK] Tabela raw.suspects restaurada (8 registros integros).
[OK] Tabela raw.evidence restaurada (20 registros integros).
[OK] Tabela raw.testimonies restaurada (14 registros integros).
[OK] Tabela raw.cameras restaurada (28 registros integros).
[OK] Tabela raw.locations restaurada (8 registros integros).
[OK] Estado inicial restaurado. Execute `dbt run` e `dbt test` para validar.

=== Reexecutando dbt run e dbt test apos restauracao ===
Done. PASS=11 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=11
Done. PASS=96 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=96


### 7. Consulta Analitica Final no DuckDB (`mart_case_summary` e Conflitos de Alibi)
Por fim, consultamos diretamente o banco `data/detectivedb.duckdb` para inspecionar a tabela consolidada `mart_case_summary` e as contradicoes de alibi detectadas em `fct_testimonies`.

In [7]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    summary_df = con.execute(
        '''
        SELECT
            investigation_priority_rank AS rank,
            suspect_id,
            suspect_name,
            occupation,
            risk_level AS initial_risk,
            evidence_count,
            inconsistent_testimony_count,
            suspicious_location_events,
            alibi_camera_conflict_count,
            contradiction_flag,
            suspicion_score
        FROM main.mart_case_summary
        ORDER BY investigation_priority_rank ASC, suspect_id ASC;
        '''
    ).df()

    conflicts_df = con.execute(
        '''
        SELECT
            testimony_id,
            suspect_id,
            suspect_name,
            claimed_event_time,
            claimed_location_name,
            conflicting_camera_refs,
            conflicting_observed_locations,
            min_conflict_diff_minutes
        FROM main.fct_testimonies
        WHERE has_alibi_camera_conflict = TRUE
        ORDER BY claimed_event_time ASC;
        '''
    ).df()

print('=== TABELA CONSOLIDADA: mart_case_summary (Score Heuristico Exploratorio) ===')
display(summary_df)

print('\n=== CONTRADICOES ESTRUTURADAS DE ALIBI (Janela <= 5 min) ===')
display(conflicts_df)